<a href="https://colab.research.google.com/github/anjalikaushik20/Problem-solving/blob/master/templates/11_sliding_window.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 Hard: Sliding Window Attention

Implement **Sliding Window Attention** — used in Longformer, Mistral, etc. for efficient long-context processing.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size).

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.6 MB/s eta 0:00:00


In [2]:
import torch
import math

In [6]:
# ✏️ YOUR IMPLEMENTATION HERE

def sliding_window_attention(Q, K, V, window_size):
  B, Tq, Dq = Q.shape
  B, Tk, Dk = K.shape
  B, Tv, Dv = V.shape

  i = torch.arange(Tq).unsqueeze(1)
  j = torch.arange(Tk).unsqueeze(0)
  mask = (i-j).abs() > window_size
  scores = (Q @ K.transpose(-2, -1))/math.sqrt(Dk)
  scores = scores.masked_fill(mask, float('-inf'))
  weights = torch.softmax(scores, dim=-1)
  output = weights @ V
  return output
    # pass  # Replace this

In [7]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
print("Output shape:", out.shape)  # (1, 6, 8)

# window=0 should return V
out0 = sliding_window_attention(Q, K, V, window_size=0)
print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

Output shape: torch.Size([1, 6, 8])
window=0 == V? True


In [8]:
from torch_judge import check
check('sliding_window')


🧪 Testing: Sliding Window Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (5.5ms)
  ✅ [2/5] window_size=0 — only sees itself (0.9ms)
  ✅ [3/5] Large window equals full attention (1.7ms)
  ✅ [4/5] Distant tokens don't affect output (3.1ms)
  ✅ [5/5] Gradient flow (22.5ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (33.7ms total)
  Progress saved. Run status() to see your dashboard.

